<div style="font-size: 1.25em; font-weight: 700;">PBMC 1k QC workflow</div>

In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

# Run from the repository root or from downstream/.
# If the repository is elsewhere, set project_dir explicitly here.
project_dir = Path.cwd()
if not (project_dir / "reference").exists() and (
    project_dir.parent / "reference"
).exists():
    project_dir = project_dir.parent

reference_dir = project_dir / "reference"
results_dir = project_dir / "downstream" / "results"
soupx_dir = project_dir / "downstream" / "soupx_input"
results_dir.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(
    reference_dir / "pbmc_1k_raw_annotated.h5ad"
)


<div style="font-size: 1.12em; font-weight: 700;">1. 数据读取与初始检查</div>

检查当前 AnnData 的基本结构、维度和原始计数状态。

In [ ]:
print("AnnData:")
print(adata)

print("\n数据形状:")
print(adata.shape)

print("\nobs字段:")
print(adata.obs.columns.tolist())

print("\nvar字段:")
print(adata.var.columns.tolist())

print("\nlayers:")
print(list(adata.layers.keys()))

print("\nX类型:")
print(type(adata.X))

print("\nX中前20个非零计数:")
print(adata.X.data[:20])

print("\nX非零计数范围:")
print("最小值:", adata.X.data.min())
print("最大值:", adata.X.data.max())
print(
    "是否全部接近整数:",
    np.allclose(adata.X.data, np.round(adata.X.data))
)

print("\n前5个barcode:")
print(adata.obs_names[:5].tolist())

print("\n前5个基因ID:")
print(adata.var_names[:5].tolist())

print("\n前5个barcode的X总计数:")
print(np.asarray(adata.X.sum(axis=1)).ravel()[:5])

print("\n前5个barcode的unspliced总计数:")
print(
    np.asarray(
        adata.layers["unspliced"].sum(axis=1)
    ).ravel()[:5]
)

<div style="font-size: 1.12em; font-weight: 700;">2. 基础 QC 指标与候选 barcode</div>

根据 gene symbol 标记线粒体、核糖体和血红蛋白基因，准备 QC 指标。

In [ ]:
gene_symbols = adata.var["gene_symbol"].astype(str)

adata.var["mt"] = gene_symbols.str.startswith("MT-")
adata.var["ribo"] = gene_symbols.str.startswith(("RPS", "RPL"))
adata.var["hb"] = gene_symbols.str.contains(
    r"^HB[ABDEGMQZ]\d*(?!\w)",
    regex=True,
)

print("mitochondrial genes:", int(adata.var["mt"].sum()))
print("ribosomal genes:", int(adata.var["ribo"].sum()))
print("hemoglobin genes:", int(adata.var["hb"].sum()))

计算每个 barcode 的总计数、检测基因数及特殊基因比例。

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    inplace=True,
    percent_top=[20],
    log1p=True,
)

汇总主要 QC 指标的分布，用于判断数据范围和异常区域。

In [ ]:
qc_columns = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
    "pct_counts_in_top_20_genes",
]

print(adata.obs[qc_columns].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T)

绘制 QC 小提琴图、散点图和直方图，观察 barcode 质量分布。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 设置图像参数
sc.set_figure_params(
    dpi=100,
    facecolor="white",
    frameon=False,
)

# 1. 绘制基本QC指标的小提琴图
sc.pl.violin(
    adata,
    keys=[
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
    ],
    jitter=0.3,
    multi_panel=True,
)

# 2. 绘制前20个高计数基因的计数占比
ax = sc.pl.violin(
    adata,
    keys="pct_counts_in_top_20_genes",
    jitter=0.3,
    show=False,
)

# 处理Scanpy可能返回多个坐标轴的情况
if isinstance(ax, (list, tuple, np.ndarray)):
    ax = np.asarray(ax).ravel()[0]

# 图中使用英文标签
ax.set_title("Fraction of counts in top 20 genes")
ax.set_ylabel("Percentage of counts (%)")

# 调整图像边距，不使用tight_layout
ax.figure.subplots_adjust(
    top=0.88,
    bottom=0.15,
    left=0.15,
)

plt.show()

# 3. 绘制总计数、检测到的基因数和线粒体比例的关系
sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
)

# 4. 绘制总计数和线粒体比例的直方图
fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4),
    constrained_layout=True,
)

# 原始尺度下的总计数
sns.histplot(
    data=adata.obs,
    x="total_counts",
    bins=60,
    ax=axes[0],
)

axes[0].set_title("Total counts")
axes[0].set_xlabel("Total counts")
axes[0].set_ylabel("Barcode count")

# 对数坐标下的总计数
sns.histplot(
    data=adata.obs,
    x="total_counts",
    bins=60,
    log_scale=True,
    ax=axes[1],
)

axes[1].set_title("Total counts on log scale")
axes[1].set_xlabel("Total counts")
axes[1].set_ylabel("Barcode count")

# 线粒体计数比例
sns.histplot(
    data=adata.obs,
    x="pct_counts_mt",
    bins=60,
    ax=axes[2],
)

axes[2].set_title("Mitochondrial percentage")
axes[2].set_xlabel("Mitochondrial counts (%)")
axes[2].set_ylabel("Barcode count")

plt.show()

用 MAD 方法标记异常 barcode，并加入线粒体比例阈值。

In [ ]:
from scipy.stats import median_abs_deviation

def is_outlier(adata, metric, nmads):
    values = adata.obs[metric].to_numpy()
    median = np.median(values)
    mad = median_abs_deviation(values)
    flag = (
        (values < median - nmads * mad)
        | (values > median + nmads * mad)
    )
    return flag, median, mad

adata.obs["outlier"] = False

for metric in [
    "log1p_total_counts",
    "log1p_n_genes_by_counts",
    "pct_counts_in_top_20_genes",
]:
    flag, median, mad = is_outlier(adata, metric, 5)
    adata.obs["outlier"] |= flag
    print(metric, "flagged:", int(flag.sum()))

mt_flag, mt_median, mt_mad = is_outlier(
    adata, "pct_counts_mt", 3
)
adata.obs["mt_outlier"] = (
    mt_flag | (adata.obs["pct_counts_mt"] > 8)
)
adata.obs["qc_candidate"] = (
    adata.obs["outlier"] | adata.obs["mt_outlier"]
)

print("base outlier:", int(adata.obs["outlier"].sum()))
print("mitochondrial outlier:", int(adata.obs["mt_outlier"].sum()))
print("QC candidates:", int(adata.obs["qc_candidate"].sum()))

列出 QC 候选 barcode，查看它们具体是哪些异常。

In [ ]:
candidate_columns = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
    "pct_counts_in_top_20_genes",
    "outlier",
    "mt_outlier",
    "qc_candidate",
]

candidate_obs = adata.obs.loc[
    adata.obs["qc_candidate"], candidate_columns
].sort_values(
    ["pct_counts_mt", "n_genes_by_counts"],
    ascending=[False, True],
)
print(candidate_obs.head(30).to_string())

保存 QC 指标和候选标记，供后续 SoupX 与 doublet 分析使用。

In [ ]:
# 保存 QC 指标和候选标记，供后续 SoupX 与 doublet 分析使用。

adata.write(
    results_dir / "pbmc_1k_qc_metrics.h5ad"
)

adata.obs.to_csv(
    results_dir / "pbmc_1k_qc_metrics.csv"
)

print("QC metrics saved.")
print("AnnData:", results_dir / "pbmc_1k_qc_metrics.h5ad")
print("QC table:", results_dir / "pbmc_1k_qc_metrics.csv")


<div style="font-size: 1.12em; font-weight: 700;">3. SoupX 输入矩阵准备</div>

读取 unfiltered 矩阵，确认它包含比 filtered 矩阵更多的 barcode。

In [ ]:
adata_raw = sc.read_h5ad(
    reference_dir / "pbmc_1k_unfiltered.h5ad"
)

print("filtered cells:", adata.n_obs)
print("raw barcodes:", adata_raw.n_obs)
assert adata_raw.n_obs > adata.n_obs


重新载入 filtered 和 raw-like 矩阵，准备构建 SoupX 输入。

In [ ]:
adata_filtered = ad.read_h5ad(
    results_dir / "pbmc_1k_qc_metrics.h5ad"
)
adata_raw = ad.read_h5ad(
    reference_dir / "pbmc_1k_unfiltered.h5ad"
)

print("filtered:", adata_filtered.shape)
print("raw-like:", adata_raw.shape)
print("filtered layers:", list(adata_filtered.layers.keys()))
print("raw-like layers:", list(adata_raw.layers.keys()))


检查两套矩阵的基因顺序、barcode 覆盖和 count 定义是否一致。

In [ ]:
assert adata_raw.n_vars == adata_filtered.n_vars

assert adata_raw.var_names.equals(
    adata_filtered.var_names
)

assert "unspliced" in adata_raw.layers
assert "unspliced" in adata_filtered.layers

missing_filtered = adata_filtered.obs_names.difference(
    adata_raw.obs_names
)

print(
    "filtered barcodes missing from raw-like:",
    len(missing_filtered),
)

print(missing_filtered.tolist())

adata_soupx_toc = adata_filtered[
    adata_filtered.obs_names.isin(
        adata_raw.obs_names
    )
].copy()

assert set(
    adata_soupx_toc.obs_names
).issubset(
    set(adata_raw.obs_names)
)

print("Gene order: OK")
print("SoupX filtered shape:", adata_soupx_toc.shape)
print("Count definition: S + A in .X")

把 gene symbol 处理成适合矩阵文件使用的唯一基因名。

In [ ]:
def make_unique_names(names):
    seen = {}
    result = []

    for name in names:
        name = str(name)
        count = seen.get(name, 0)

        if count == 0:
            result.append(name)
        else:
            result.append(f"{name}.{count}")

        seen[name] = count + 1

    return pd.Index(result)


gene_names = adata_raw.var["gene_symbol"].copy()
gene_names = gene_names.fillna(
    pd.Series(
        adata_raw.var_names,
        index=adata_raw.var_names,
    )
)
gene_names = gene_names.astype(str)

gene_names = pd.Index([
    gene_id if name in {"", "nan", "None"} else name
    for gene_id, name in zip(
        adata_raw.var_names,
        gene_names,
    )
])

gene_names = make_unique_names(gene_names)

print("unique gene names:", gene_names.is_unique)

导出 SoupX 所需的 raw、filtered、基因和 barcode 文件。

In [ ]:
from scipy.io import mmwrite

soupx_dir.mkdir(parents=True, exist_ok=True)

# 使用 .X，也就是 S+A；不要使用 unspliced layer。
raw_matrix = adata_raw.X.T.tocsc()
filtered_matrix = adata_soupx_toc.X.T.tocsc()

mmwrite(soupx_dir / "raw_SA.mtx", raw_matrix)
mmwrite(soupx_dir / "filtered_SA.mtx", filtered_matrix)

pd.Series(gene_names).to_csv(
    soupx_dir / "genes.tsv",
    index=False,
    header=False,
)

pd.Series(adata_raw.obs_names).to_csv(
    soupx_dir / "raw_barcodes.tsv",
    index=False,
    header=False,
)

pd.Series(adata_soupx_toc.obs_names).to_csv(
    soupx_dir / "filtered_barcodes.tsv",
    index=False,
    header=False,
)

print("raw matrix:", raw_matrix.shape)
print("filtered matrix:", filtered_matrix.shape)


<div style="font-size: 1.12em; font-weight: 700;">4. SoupX 初步聚类与环境 RNA 校正</div>

在 filtered barcode 上做初步聚类，为 SoupX 估计环境 RNA 提供 cluster 信息。

In [ ]:
import scanpy as sc

# 创建聚类专用副本
adata_cluster = adata_soupx_toc.copy()

# 去掉极少表达的基因
sc.pp.filter_genes(
    adata_cluster,
    min_cells=3,
)

print("genes after filtering:", adata_cluster.n_vars)

# 先归一化和log转换
sc.pp.normalize_total(
    adata_cluster,
    target_sum=1e4,
)
sc.pp.log1p(adata_cluster)

# 再选择高变基因
sc.pp.highly_variable_genes(
    adata_cluster,
    n_top_genes=2000,
    flavor="seurat",
    subset=True,
)

# 标准化
sc.pp.scale(
    adata_cluster,
    max_value=10,
)

# PCA
sc.tl.pca(
    adata_cluster,
    svd_solver="arpack",
)

# 邻居图
sc.pp.neighbors(
    adata_cluster,
    n_neighbors=15,
)

# Leiden聚类
sc.tl.leiden(
    adata_cluster,
    resolution=0.5,
    key_added="soupx_cluster",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=123,
)

print(
    adata_cluster.obs["soupx_cluster"].value_counts()
)

保存初步 cluster 标签，供 SoupX 脚本读取。

In [ ]:
clusters = adata_cluster.obs[["soupx_cluster"]].copy()
clusters.index.name = "barcode"

soupx_dir.mkdir(parents=True, exist_ok=True)
clusters.to_csv(
    soupx_dir / "filtered_clusters.tsv",
    sep="	",
)

print("saved:", soupx_dir / "filtered_clusters.tsv")
print(clusters.shape)


读取 SoupX 校正矩阵，并将校正后的 S+A counts 放回 AnnData。

In [ ]:
from scipy.io import mmread

corrected_path = results_dir / "pbmc_1k_soupx_corrected.mtx"

corrected_gene_by_cell = mmread(corrected_path)
corrected_cell_by_gene = corrected_gene_by_cell.T.tocsr()

adata_soupx = adata_soupx_toc.copy()

adata_soupx.obs["qc_candidate_pre_soupx"] = (
    adata_soupx.obs["qc_candidate"]
)

adata_soupx.X = corrected_cell_by_gene

print(adata_soupx.shape)
print(adata_soupx.X.sum())

adata_soupx.write_h5ad(
    reference_dir / "pbmc_1k_soupx_corrected.h5ad"
)


比较 SoupX 校正前后的基因计数，检查校正影响。

In [ ]:
before = np.asarray(
    adata_soupx_toc.X.sum(axis=0)
).ravel()

after = np.asarray(
    adata_soupx.X.sum(axis=0)
).ravel()

comparison = pd.DataFrame({
    "gene_id": adata_soupx_toc.var_names,
    "before": before,
    "after": after,
})

comparison["removed"] = (
    comparison["before"] - comparison["after"]
)

print(comparison.sort_values(
    "removed",
    ascending=False,
).head(20))

<div style="font-size: 1.12em; font-weight: 700;">5. SoupX 后 QC 复核</div>

在 SoupX 校正后的 counts 上重新计算 QC，并保存新的 QC 字段。

In [ ]:
gene_symbols = adata_soupx.var["gene_symbol"].astype(str)

adata_soupx.var["mt"] = gene_symbols.str.startswith("MT-")
adata_soupx.var["ribo"] = gene_symbols.str.startswith(("RPS", "RPL"))
adata_soupx.var["hb"] = gene_symbols.str.contains(
    r"^HB[ABDEGMQZ]\d*(?!\w)",
    regex=True,
)

sc.pp.calculate_qc_metrics(
    adata_soupx,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=[20],
    log1p=True,
    inplace=True,
)

print(
    adata_soupx.obs[
        [
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
        ]
    ].describe().T
)

# 在校正后的指标上重新生成 MAD 候选标记。
from scipy.stats import median_abs_deviation

def is_outlier_post_soupx(adata, metric, nmads):
    values = adata.obs[metric].to_numpy()
    median = np.median(values)
    mad = median_abs_deviation(values)
    return (
        (values < median - nmads * mad)
        | (values > median + nmads * mad)
    )

adata_soupx.obs["outlier_post_soupx"] = False

for metric in [
    "log1p_total_counts",
    "log1p_n_genes_by_counts",
    "pct_counts_in_top_20_genes",
]:
    adata_soupx.obs["outlier_post_soupx"] |= (
        is_outlier_post_soupx(adata_soupx, metric, 5)
    )

adata_soupx.obs["mt_outlier_post_soupx"] = (
    is_outlier_post_soupx(
        adata_soupx,
        "pct_counts_mt",
        3,
    )
    | (adata_soupx.obs["pct_counts_mt"] > 8)
)

adata_soupx.obs["qc_candidate_post_soupx"] = (
    adata_soupx.obs["outlier_post_soupx"]
    | adata_soupx.obs["mt_outlier_post_soupx"]
)

print(
    "post-SoupX QC candidates:",
    int(adata_soupx.obs["qc_candidate_post_soupx"].sum()),
)

adata_soupx.write_h5ad(
    reference_dir / "pbmc_1k_soupx_corrected.h5ad"
)


<div style="font-size: 1.12em; font-weight: 700;">6. Doublet 检测与人工复核</div>

读取校正后的对象，并合并 scDblFinder 的 singlet/doublet 结果。

In [ ]:
adata = sc.read_h5ad(
    reference_dir / "pbmc_1k_soupx_corrected.h5ad"
)

dbl = pd.read_csv(
    results_dir / "scDblFinder_results.tsv",
    sep="	",
).set_index("barcode")

adata.obs = adata.obs.join(
    dbl,
    how="left",
    validate="one_to_one",
)

print(adata.shape)
print(adata.obs["class"].value_counts())


比较 singlet 和 doublet 的 QC 指标中位数。

In [ ]:
qc_cols = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
]

print(
    adata.obs.groupby("class")[qc_cols].median()
)

用小提琴图和散点图观察 singlet 与 doublet 的质量差异。

In [ ]:
sc.pl.violin(
    adata,
    qc_cols,
    groupby="class",
    multi_panel=True,
    jitter=0.4
)

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="class"
)

统计 singlet 和 doublet 中同时具有高 counts、高检测基因数的 barcode。

In [ ]:
singlet = adata.obs[
    adata.obs["class"] == "singlet"
]

doublet = adata.obs[
    adata.obs["class"] == "doublet"
]

count_cutoff = singlet["total_counts"].quantile(0.75)
gene_cutoff = singlet["n_genes_by_counts"].quantile(0.75)

for label, group in [
    ("singlet", singlet),
    ("doublet", doublet),
]:
    upper_right = (
        (group["total_counts"] > count_cutoff)
        & (group["n_genes_by_counts"] > gene_cutoff)
    )
    print(
        label,
        int(upper_right.sum()),
        "/",
        len(group),
        "=",
        round(upper_right.mean(), 3),
    )

查看 doublet 的 cluster 来源及 mostLikelyOrigin 结果。

In [ ]:
cluster_table = pd.read_csv(
    soupx_dir / "filtered_clusters.tsv",
    sep="	",
).set_index("barcode")

adata.obs["soupx_cluster"] = (
    cluster_table.reindex(adata.obs_names)["soupx_cluster"]
    .astype(str)
    .to_numpy()
)

scdbl_audit = pd.read_csv(
    results_dir / "scDblFinder_full_table.tsv",
    sep="	",
).set_index("barcode")

doublet_barcodes = adata.obs_names[
    adata.obs["class"] == "doublet"
]

print(
    scdbl_audit.reindex(doublet_barcodes)[
        ["mostLikelyOrigin", "originAmbiguous"]
    ].value_counts()
)


用 T、B、NK 和髓系 marker 对 singlet/doublet 做表达评分。

In [ ]:
adata_marker = adata.copy()

sc.pp.normalize_total(
    adata_marker,
    target_sum=1e4,
)
sc.pp.log1p(adata_marker)

marker_sets = {
    "T": ["CD3D", "CD3E", "TRBC1", "TRBC2", "IL7R", "LTB"],
    "B": ["MS4A1", "CD79A", "CD37", "CD74", "HLA-DRA", "CD79B"],
    "NK": ["NKG7", "GNLY", "PRF1", "GZMB", "KLRD1"],
    "myeloid": ["LYZ", "S100A8", "S100A9", "LST1", "TYROBP", "CTSS"],
}

gene_symbols = adata_marker.var["gene_symbol"].astype(str)

def symbols_to_gene_ids(symbols):
    ids = []
    for symbol in symbols:
        ids.extend(
            adata_marker.var_names[
                gene_symbols == symbol
            ].tolist()
        )
    return list(dict.fromkeys(ids))

score_cols = []

for name, symbols in marker_sets.items():
    gene_ids = symbols_to_gene_ids(symbols)
    print(name, "mapped genes:", len(gene_ids))

    sc.tl.score_genes(
        adata_marker,
        gene_list=gene_ids,
        score_name=f"{name}_score",
        use_raw=False,
    )
    score_cols.append(f"{name}_score")

print(
    adata_marker.obs.groupby("class")[score_cols].median()
)

确认 marker 注释列可用，再进行后续双细胞 marker 可视化。

In [ ]:
print(adata.var[["gene_symbol"]].head())

整理真实 gene symbol，并绘制疑似 doublet 的 marker 热图。

In [ ]:
gene_symbols = adata_marker.var["gene_symbol"].astype(str)

marker_symbols = []

for symbols in marker_sets.values():
    marker_symbols.extend(
        [
            symbol
            for symbol in symbols
            if (gene_symbols == symbol).any()
        ]
    )

marker_symbols = list(dict.fromkeys(marker_symbols))

doublet_view = adata_marker[
    adata_marker.obs["class"] == "doublet"
].copy()

doublet_view.obs["doublet_barcode"] = (
    doublet_view.obs_names
)

sc.pl.heatmap(
    doublet_view,
    var_names=marker_symbols,
    groupby="doublet_barcode",
    gene_symbols="gene_symbol",
    use_raw=False,
    standard_scale="var",
    swap_axes=True,
    figsize=(10, 14),
)

交叉统计 cluster 与 singlet/doublet 类别，检查是否存在 doublet 富集 cluster。

In [ ]:
print(
    pd.crosstab(
        adata.obs["soupx_cluster"],
        adata.obs["class"]
    )
)

<div style="font-size: 1.12em; font-weight: 700;">7. reviewed 对象与重新聚类</div>

In [ ]:
# 只去除 scDblFinder 判定的 doublet。
# SoupX precluster 仅用于环境 RNA 估计，不作为细胞质量标签。
# precluster 7 会在独立聚类、marker 和 QC 指标中复核。
adata_reviewed = adata[
    adata.obs["class"] == "singlet"
].copy()

pre7 = adata_reviewed.obs["soupx_cluster"] == "7"
print("reviewed object:", adata_reviewed.shape)
print(
    "precluster 7 singlets retained for independent review:",
    int(pre7.sum()),
)


In [ ]:
adata_analysis = adata_reviewed.copy()

sc.pp.normalize_total(
    adata_analysis,
    target_sum=1e4,
)

sc.pp.log1p(adata_analysis)

sc.pp.highly_variable_genes(
    adata_analysis,
    n_top_genes=2000,
    flavor="seurat",
)

sc.pp.scale(
    adata_analysis,
    max_value=10,
)

sc.tl.pca(
    adata_analysis,
    use_highly_variable=True,
    svd_solver="arpack",
)

sc.pp.neighbors(
    adata_analysis,
    n_neighbors=15,
    n_pcs=30,
)

sc.tl.umap(
    adata_analysis,
    random_state=123,
)

sc.tl.leiden(
    adata_analysis,
    resolution=0.5,
    key_added="reviewed_cluster",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=123,
)

print(adata_analysis.shape)
print(
    adata_analysis.obs["reviewed_cluster"]
    .value_counts()
    .sort_index()
)

<div style="font-size: 1.12em; font-weight: 700;">8. Cluster marker review</div>


In [ ]:
adata_marker_reviewed = adata_reviewed.copy()

adata_marker_reviewed.obs["reviewed_cluster"] = (
    adata_analysis.obs["reviewed_cluster"]
    .reindex(adata_marker_reviewed.obs_names)
    .astype(str)
    .to_numpy()
)

sc.pp.normalize_total(
    adata_marker_reviewed,
    target_sum=1e4,
)
sc.pp.log1p(adata_marker_reviewed)

sc.tl.rank_genes_groups(
    adata_marker_reviewed,
    groupby="reviewed_cluster",
    method="wilcoxon",
    use_raw=False,
    key_added="reviewed_markers",
)

gene_symbols = adata_marker_reviewed.var["gene_symbol"].astype("string")
fallback_symbols = pd.Series(
    adata_marker_reviewed.var_names.astype(str),
    index=adata_marker_reviewed.var_names,
    dtype="string",
)
gene_symbols = gene_symbols.fillna(fallback_symbols)
gene_id_to_symbol = pd.Series(
    gene_symbols.to_numpy(),
    index=adata_marker_reviewed.var_names,
)

marker_table = sc.get.rank_genes_groups_df(
    adata_marker_reviewed,
    group=None,
    key="reviewed_markers",
)
marker_table["gene_symbol"] = (
    marker_table["names"]
    .astype(str)
    .map(gene_id_to_symbol)
    .fillna(marker_table["names"].astype(str))
)
marker_table["rank"] = (
    marker_table.groupby("group", sort=False)
    .cumcount()
    .add(1)
)

top_markers = (
    marker_table[
        ["group", "rank", "gene_symbol", "scores", "pvals_adj"]
    ]
    .groupby("group", sort=False)
    .head(10)
)

print(top_markers.to_string(index=False))

# 检查原 SoupX precluster 是否在独立聚类中形成稳定、可解释的群。
print("\nprecluster vs independent reviewed_cluster:")
print(
    pd.crosstab(
        adata_marker_reviewed.obs["reviewed_cluster"],
        adata_marker_reviewed.obs["soupx_cluster"],
    )
)

sc.pl.rank_genes_groups(
    adata_marker_reviewed,
    n_genes=10,
    sharey=False,
    key="reviewed_markers",
    gene_symbols="gene_symbol",
)


In [ ]:
cluster_to_celltype = {
    "0": "HLA-DRA+ antigen-presenting",
    "1": "Classical monocyte",
    "2": "Plasma cell-like",
    "3": "B cell",
    "4": "CD4 T-like",
    "5": "Cytotoxic T/NKT-like",
    "6": "NK cell",
    "7": "MT-high / low-quality candidate",
}

adata_analysis.obs["cell_type_provisional"] = (
    adata_analysis.obs["reviewed_cluster"]
    .astype(str)
    .map(cluster_to_celltype)
    .astype("category")
)

adata_marker_reviewed.obs["cell_type_provisional"] = (
    adata_analysis.obs["cell_type_provisional"]
    .reindex(adata_marker_reviewed.obs_names)
    .astype("category")
)

print(
    adata_analysis.obs["cell_type_provisional"]
    .value_counts()
)

In [ ]:
sc.pl.umap(
    adata_analysis,
    color=[
        "reviewed_cluster",
        "cell_type_provisional",
        "total_counts",
        "n_genes_by_counts",
    ],
    frameon=False,
)

In [ ]:
marker_panel = [
    "CST3", "CD74", "HLA-DRA", "HLA-DPA1",
    "LYZ", "S100A8", "S100A9", "S100A12", "VCAN", "FCN1",
    "JCHAIN", "MZB1", "TNFRSF17", "IGHA1",
    "LST1", "FCGR3A", "AIF1", "MS4A7",
    "MS4A1", "CD79A", "CD79B", "CD37",
    "CD3D", "CD3E", "TRAC", "IL7R", "TCF7", "BCL11B",
    "CD8A", "CD8B",
    "NKG7", "KLRD1", "GNLY", "CTSW", "PRF1",
    "MT-CO1", "MT-CO3", "MT-ND1", "MT-ND4", "MT-CYB",
]

available_symbols = set(
    adata_marker_reviewed.var["gene_symbol"].astype(str)
)

marker_panel = [
    gene for gene in marker_panel
    if gene in available_symbols
]

sc.pl.dotplot(
    adata_marker_reviewed,
    var_names=marker_panel,
    groupby="cell_type_provisional",
    gene_symbols="gene_symbol",
    use_raw=False,
    standard_scale="var",
    figsize=(14, 7),
)

In [ ]:
# 将独立复核结果写回 corrected-count 对象，并移除有独立证据支持的 MT-high cluster。

adata_reviewed.obs["reviewed_cluster"] = (
    adata_analysis.obs["reviewed_cluster"]
    .reindex(adata_reviewed.obs_names)
    .astype(str)
    .to_numpy()
)

adata_reviewed.obs["cell_type"] = (
    adata_analysis.obs["cell_type_provisional"]
    .reindex(adata_reviewed.obs_names)
    .astype(str)
    .to_numpy()
)

keep_cells = (
    adata_reviewed.obs["cell_type"]
    != "MT-high / low-quality candidate"
)
adata_final = adata_reviewed[keep_cells].copy()

print("adata_final:", adata_final.shape)
print(adata_final.obs["cell_type"].value_counts())


In [ ]:
adata_final_analysis = adata_final.copy()

sc.pp.normalize_total(
    adata_final_analysis,
    target_sum=1e4,
)

sc.pp.log1p(adata_final_analysis)

sc.pp.highly_variable_genes(
    adata_final_analysis,
    n_top_genes=2000,
    flavor="seurat",
)

sc.pp.scale(
    adata_final_analysis,
    max_value=10,
)

sc.tl.pca(
    adata_final_analysis,
    use_highly_variable=True,
    svd_solver="arpack",
)

sc.pp.neighbors(
    adata_final_analysis,
    n_neighbors=15,
    n_pcs=30,
)

sc.tl.umap(
    adata_final_analysis,
    random_state=123,
)

sc.tl.leiden(
    adata_final_analysis,
    resolution=0.5,
    key_added="final_cluster",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=123,
)

print(adata_final_analysis.shape)
print(
    adata_final_analysis.obs["final_cluster"]
    .value_counts()
    .sort_index()
)

In [ ]:
sc.pl.umap(
    adata_final_analysis,
    color=[
        "final_cluster",
        "cell_type",
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
    ],
    frameon=False,
)

In [ ]:
cluster_composition = pd.crosstab(
    adata_final_analysis.obs["final_cluster"],
    adata_final_analysis.obs["cell_type"],
    normalize="index",
).round(3)

print(cluster_composition)

In [ ]:
qc_summary = (
    adata_final_analysis.obs
    .groupby("cell_type", observed=True)
    .agg(
        n_cells=("cell_type", "size"),
        median_counts=("total_counts", "median"),
        median_genes=("n_genes_by_counts", "median"),
        median_mt=("pct_counts_mt", "median"),
        max_mt=("pct_counts_mt", "max"),
    )
    .round(3)
)

print(qc_summary)

print("\nTop 10 mitochondrial cells:")
print(
    adata_final_analysis.obs.nlargest(
        10,
        "pct_counts_mt",
    )[
        [
            "cell_type",
            "final_cluster",
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
        ]
    ]
)

In [ ]:
adata_final.obs["final_cluster"] = (
    adata_final_analysis.obs["final_cluster"]
    .reindex(adata_final.obs_names)
    .astype(str)
    .to_numpy()
)

print(
    adata_final.obs[
        ["cell_type", "final_cluster"]
    ].head()
)

In [ ]:
# 只标记，不修改 adata_final

adata_final.obs["qc_recheck"] = (
    (adata_final.obs["pct_counts_mt"] >= 50)
    |
    (
        (adata_final.obs["pct_counts_mt"] >= 25)
        & (adata_final.obs["n_genes_by_counts"] < 500)
    )
    |
    (adata_final.obs["n_genes_by_counts"] < 200)
)

print(
    adata_final.obs["qc_recheck"].value_counts()
)

print("\nQC candidates:")
print(
    adata_final.obs.loc[
        adata_final.obs["qc_recheck"],
        [
            "cell_type",
            "final_cluster",
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
        ],
    ]
    .sort_values("pct_counts_mt", ascending=False)
)

In [ ]:
sc.pl.violin(
    adata_final,
    keys=[
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
    ],
    groupby="cell_type",
    rotation=45,
    stripplot=False,
)

sc.pl.scatter(
    adata_final,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
)

In [ ]:
adata_final_marker = adata_final.copy()

adata_final_marker.obs["final_cluster"] = (
    adata_final_analysis.obs["final_cluster"]
    .reindex(adata_final_marker.obs_names)
    .astype(str)
    .to_numpy()
)

sc.pp.normalize_total(
    adata_final_marker,
    target_sum=1e4,
)

sc.pp.log1p(adata_final_marker)

sc.tl.rank_genes_groups(
    adata_final_marker,
    groupby="final_cluster",
    method="wilcoxon",
    use_raw=False,
    key_added="final_markers",
)

In [ ]:
gene_symbols = (
    adata_final_marker.var["gene_symbol"]
    .astype("string")
)

fallback_symbols = pd.Series(
    adata_final_marker.var_names.astype(str),
    index=adata_final_marker.var_names,
    dtype="string",
)

gene_symbols = gene_symbols.fillna(fallback_symbols)

gene_id_to_symbol_final = pd.Series(
    gene_symbols.to_numpy(),
    index=adata_final_marker.var_names,
)

final_marker_table = sc.get.rank_genes_groups_df(
    adata_final_marker,
    group=None,
    key="final_markers",
)

final_marker_table["gene_symbol"] = (
    final_marker_table["names"]
    .astype(str)
    .map(gene_id_to_symbol_final)
    .fillna(final_marker_table["names"].astype(str))
)

final_marker_table["group"] = (
    final_marker_table["group"].astype(str)
)

print(
    final_marker_table[
        final_marker_table["group"].isin(["3", "6"])
    ][
        [
            "group",
            "gene_symbol",
            "scores",
            "pvals_adj",
        ]
    ]
    .groupby("group", sort=False)
    .head(20)
    .to_string(index=False)
)

In [ ]:
marker_panel_final = [
    "LYZ", "S100A8", "S100A9", "S100A12", "VCAN", "FCN1",
    "LST1", "FCGR3A", "MS4A7", "AIF1",
    "CD3D", "CD3E", "TRAC", "IL7R", "TCF7",
    "CD8A", "CD8B",
    "MS4A1", "CD79A", "CD74",
    "NKG7", "KLRD1", "GNLY",
]

available_symbols = set(
    adata_final_marker.var["gene_symbol"].astype(str)
)

marker_panel_final = [
    gene for gene in marker_panel_final
    if gene in available_symbols
]

sc.pl.dotplot(
    adata_final_marker,
    var_names=marker_panel_final,
    groupby="final_cluster",
    gene_symbols="gene_symbol",
    use_raw=False,
    standard_scale="var",
    figsize=(13, 7),
)

In [ ]:
adata_qc_final = adata_final[
    ~adata_final.obs["qc_recheck"]
].copy()

print("after final cell-level QC:", adata_qc_final.shape)
print(adata_qc_final.obs["cell_type"].value_counts())

# 审阅意见要求补充最终的 gene-level filtering。
# 只保留至少在 20 个细胞中检测到的基因；不对 counts 做除法。
genes_before_min_cells = adata_qc_final.n_vars
sc.pp.filter_genes(adata_qc_final, min_cells=20)
print("genes before min_cells=20:", genes_before_min_cells)
print("genes after min_cells=20:", adata_qc_final.n_vars)

<div style="font-size: 1.12em; font-weight: 700;">9. Final QC sanity check</div>


在最终过滤后的细胞上进行临时归一化、降维和聚类，仅用于确认 QC 后的数据结构是否合理。

这些 normalized values 不作为后续正式分析的输入；最终保存的对象仍保留 SoupX-corrected counts。


In [ ]:
adata_qc_analysis = adata_qc_final.copy()

sc.pp.normalize_total(
    adata_qc_analysis,
    target_sum=1e4,
)

sc.pp.log1p(adata_qc_analysis)

sc.pp.highly_variable_genes(
    adata_qc_analysis,
    n_top_genes=2000,
    flavor="seurat",
)

sc.pp.scale(
    adata_qc_analysis,
    max_value=10,
)

sc.tl.pca(
    adata_qc_analysis,
    use_highly_variable=True,
    svd_solver="arpack",
)

sc.pp.neighbors(
    adata_qc_analysis,
    n_neighbors=15,
    n_pcs=30,
)

sc.tl.umap(
    adata_qc_analysis,
    random_state=123,
)

sc.tl.leiden(
    adata_qc_analysis,
    resolution=0.5,
    key_added="final_cluster_qc",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=123,
)

print(adata_qc_analysis.shape)
print(
    adata_qc_analysis.obs["final_cluster_qc"]
    .value_counts()
    .sort_index()
)

In [ ]:
sc.pl.umap(
    adata_qc_analysis,
    color=[
        "final_cluster_qc",
        "cell_type",
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
    ],
    frameon=False,
)

In [ ]:
cluster_composition_qc = pd.crosstab(
    adata_qc_analysis.obs["final_cluster_qc"],
    adata_qc_analysis.obs["cell_type"],
    normalize="index",
).round(3)

print(cluster_composition_qc)

In [ ]:
adata_qc_marker = adata_qc_final.copy()

adata_qc_marker.obs["final_cluster_qc"] = (
    adata_qc_analysis.obs["final_cluster_qc"]
    .reindex(adata_qc_marker.obs_names)
    .astype(str)
    .to_numpy()
)

sc.pp.normalize_total(
    adata_qc_marker,
    target_sum=1e4,
)

sc.pp.log1p(adata_qc_marker)

sc.tl.rank_genes_groups(
    adata_qc_marker,
    groupby="final_cluster_qc",
    method="wilcoxon",
    use_raw=False,
    key_added="final_markers_qc",
)

In [ ]:
sc.pl.rank_genes_groups(
    adata_qc_marker,
    n_genes=10,
    key="final_markers_qc",
    gene_symbols="gene_symbol",
    sharey=False,
)

<div style="font-size: 1.12em; font-weight: 700;">10. QC audit and output</div>


In [ ]:
from pathlib import Path
import pandas as pd

# 以原始 SoupX-corrected 对象中的所有 barcode 建立审计表。
audit = adata.obs.copy()

audit["removed_doublet"] = (
    audit["class"] == "doublet"
)

audit["removed_precluster7"] = False
audit["precluster7_retained_after_review"] = (
    (audit["soupx_cluster"].astype(str) == "7")
    & (~audit["removed_doublet"])
)

audit["removed_mthigh_cluster"] = (
    audit.index.isin(adata_reviewed.obs_names)
    & ~audit.index.isin(adata_final.obs_names)
)

cell_qc_candidates = set(
    adata_final.obs_names[
        adata_final.obs["qc_recheck"]
    ]
)
audit["removed_cell_qc"] = audit.index.isin(cell_qc_candidates)
audit["qc_keep"] = audit.index.isin(adata_qc_final.obs_names)

audit["remove_reason"] = "kept"
audit.loc[audit["removed_doublet"], "remove_reason"] = (
    "scDblFinder doublet"
)
audit.loc[audit["removed_mthigh_cluster"], "remove_reason"] = (
    "MT-high cluster review"
)
audit.loc[audit["removed_cell_qc"], "remove_reason"] = (
    "extreme cell-level QC"
)

print(audit["remove_reason"].value_counts())


In [ ]:
assert (
    audit["remove_reason"].value_counts()["kept"]
    == adata_qc_final.n_obs
)

assert adata_qc_final.obs["class"].eq("singlet").all()
assert (~adata_qc_final.obs["qc_recheck"]).all()
assert (adata_qc_final.var["n_cells_by_counts"] >= 20).all()

print("QC audit passed.")
print("Final object:", adata_qc_final.shape)
print(
    "precluster 7 singlets retained before final cell-level QC:",
    int(audit["precluster7_retained_after_review"].sum()),
)
print(
    "precluster 7 singlets in final object:",
    int(
        (
            audit["precluster7_retained_after_review"]
            & audit["qc_keep"]
        ).sum()
    ),
)


In [ ]:
import anndata

anndata.settings.allow_write_nullable_strings = True

filtered_path = reference_dir / "pbmc_1k_qc_filtered.h5ad"
audit_path = results_dir / "qc_filter_audit.tsv"

adata_qc_final.write_h5ad(filtered_path)
audit.to_csv(audit_path, sep="	")

print("saved:", filtered_path)
print("saved:", audit_path)
